Here's the complete architecture in **Markdown** with **LangChain** included as the orchestration framework. This version is interview-oriented and explains the end-to-end design clearly.

# AutoShift AI Agent - End-to-End System Architecture (Interview Ready)

## Project Overview

The AutoShift AI platform automates shift-related email requests using **Generative AI, LangChain, Amazon Bedrock, Qdrant, RabbitMQ, and .NET APIs**.

Instead of manually reading emails and performing shift operations, the AI system:

- Understands the email
- Identifies the user's intent
- Extracts structured information
- Validates data using RAG
- Executes the appropriate business action
- Handles missing information using Human-in-the-Loop (HITL)

The solution is designed as an **event-driven, multi-tenant, AI-powered workflow**.

---

# High-Level Architecture

```text
                          Customer

                             │

                      Sends Email

                             │

                             ▼

                     RabbitMQ Queue

                             │

                             ▼

                Python AI Consumer Service

                             │

                     Email Parsing Layer

                             │

                             ▼

                  LangChain Orchestration

                             │

        ┌────────────────────┼─────────────────────┐
        │                    │                     │
        ▼                    ▼                     ▼

 System Prompt        Amazon Bedrock         Structured Output
 (Business Rules)        (Claude)              (JSON Response)

                             │

                             ▼

                  Intent Classification

                             │

                             ▼

                RAG Validation (Qdrant)

                             │

            Semantic Search + Metadata Filter

                             │

                             ▼

                    Decision Engine

                             │

          ┌──────────────────┼─────────────────┐
          ▼                  ▼                 ▼

      Create Shift      Update Shift     Cancel Shift
         API               API              API

                             │

                             ▼

                   Response / Success

                             │

                Missing Information?

                     Yes             No

                      │

                      ▼

               Human-in-the-Loop

                      │

             Clarification Email

                      │

               Customer Replies

                      │

                 RabbitMQ Queue

                      │

                      ▼

             Resume Previous State
```

---

# Technology Stack

| Layer | Technology |
|--------|------------|
| Messaging | RabbitMQ |
| AI Service | Python |
| AI Framework | LangChain |
| LLM | Amazon Bedrock (Claude) |
| Embedding Model | Amazon Titan Embeddings |
| Vector Database | Qdrant |
| Backend APIs | ASP.NET Core (.NET) |
| Human Interaction | Email |
| State Management | Qdrant |
| Prompt Engineering | LangChain PromptTemplate |
| Output Parsing | LangChain Structured Output |

---

# End-to-End Workflow

## Step 1 - Email Reception

A customer sends an email requesting a shift operation.

Example

```text
Please create two caregivers tomorrow from 9 AM to 5 PM for ABC Hospital.
```

The email is published to **RabbitMQ**.

---

## Step 2 - RabbitMQ

RabbitMQ decouples email reception from AI processing.

### Responsibilities

- Reliable messaging
- Retry mechanism
- Asynchronous processing
- Scalability

RabbitMQ message contains:

- Subject
- Email Body
- Sender
- Thread ID
- Tenant ID
- Attachments (optional)

---

## Step 3 - AI Consumer Service

A Python service continuously listens to RabbitMQ.

Responsibilities:

- Consume messages
- Parse email
- Remove HTML
- Extract plain text
- Extract thread details
- Prepare AI request

Output

```text
Clean Email Content
```

---

# Step 4 - LangChain Orchestration

LangChain acts as the orchestration framework.

It manages:

- Prompt Templates
- Bedrock Integration
- Structured Output Parsing
- RAG Pipeline
- Tool Calling
- Decision Flow

Architecture

```text
Email

↓

LangChain

↓

Prompt Template

↓

Amazon Bedrock

↓

Structured Output

↓

Decision
```

---

# Step 5 - Prompt Engineering

LangChain creates the final prompt.

## System Prompt

Defines:

- AI role
- Business rules
- Supported operations
- Output schema
- Validation rules

Example

```text
You are an AutoShift Scheduling Assistant.

Identify the user's intent.

Return only JSON.

Never guess missing values.
```

---

## User Prompt

Contains:

- Email Subject
- Email Body
- Sender
- Previous Conversation (if applicable)

---

# Step 6 - LLM Processing

Amazon Bedrock (Claude) performs two tasks simultaneously.

## 1. Intent Detection

Possible intents

- Create Shift
- Update Shift
- Cancel Shift
- Complete Shift
- Withdraw Shift
- Restart Shift

---

## 2. Structured Output

Using LangChain Structured Output.

Example

```json
{
  "intent": "CreateShift",
  "client_name": "ABC Hospital",
  "location_name": "Pune",
  "qualification": "Caregiver",
  "shift_date": "2026-08-05",
  "start_time": "09:00",
  "end_time": "17:00",
  "shift_count": 2
}
```

Using structured output avoids unreliable text parsing.

---

# Step 7 - RAG Validation

The extracted entities are validated before executing business logic.

The AI **never directly trusts the LLM output**.

Validation includes:

- Client
- Location
- Qualification

---

## Master Data Source

Master data comes from existing .NET APIs.

Examples

- Client Master
- Location Master
- Qualification Master

These are periodically synchronized to **Qdrant**.

---

## Qdrant Vector Database

Qdrant stores embeddings for:

- Client Name
- Delivery Location
- Qualification

Metadata stored:

```text
Tenant ID

Client ID

Location ID

Qualification ID
```

---

## Retrieval Flow

```text
Client Name

↓

Embedding

↓

Semantic Search

↓

Metadata Filter

↓

Best Match
```

---

## Metadata Filtering

Since the application is multi-tenant,

every search applies

```text
Tenant ID

+

Client Name
```

This guarantees tenant isolation.

---

# Step 8 - Decision Engine

After validation,

the Decision Engine determines which backend API should be called.

Example

```text
CreateShift

↓

Create Shift API
```

```text
CancelShift

↓

Cancel Shift API
```

```text
RestartShift

↓

Restart Shift API
```

---

# Step 9 - Tool Calling

LangChain invokes the appropriate tool based on the detected intent.

Example

```text
Intent

↓

CreateShift

↓

CreateShift Tool

↓

.NET API

↓

Shift Created
```

Each tool internally calls the corresponding .NET API.

---

# Step 10 - Human-in-the-Loop (HITL)

If mandatory fields are missing,

the AI **does not guess**.

Example

Customer Email

```text
Please create caregiver tomorrow.
```

Missing

- Location
- Time
- Qualification

Workflow

```text
LLM

↓

Missing Fields

↓

Generate Clarification Email

↓

Customer Reply

↓

RabbitMQ

↓

Resume Processing
```

---

# Step 11 - State Management

Every email conversation is treated as a **Thread**.

State stored in Qdrant

- Thread ID
- Intent
- Current State
- Missing Fields
- Extracted JSON
- Processing Status

Example

```text
Thread-1001

↓

WaitingForUser
```

When the customer replies,

the AI restores the previous context instead of starting from scratch.

---

# Complete Workflow

```text
Customer Email

↓

RabbitMQ

↓

Python AI Consumer

↓

Email Parsing

↓

LangChain

↓

Prompt Template

↓

Amazon Bedrock (Claude)

↓

Intent Detection

↓

Structured Output

↓

RAG

↓

Qdrant

↓

Semantic Search

↓

Metadata Filtering

↓

Decision Engine

↓

Tool Calling

↓

.NET API

↓

Shift Created
```

---

# Human-in-the-Loop Flow

```text
LLM

↓

Missing Information

↓

Clarification Email

↓

Customer Reply

↓

RabbitMQ

↓

Restore Thread

↓

Resume Workflow
```

---

# Why LangChain?

LangChain simplifies orchestration by providing:

- Prompt Management
- Bedrock Integration
- Structured Output
- Tool Calling
- RAG Pipeline
- Retriever Integration
- Output Parsing
- Retry & Error Handling

Without LangChain, each of these capabilities would require custom implementation.

---

# Why Qdrant?

Qdrant is used for:

- Semantic Search
- Vector Storage
- Metadata Filtering
- Multi-Tenant Retrieval
- Conversation State
- Thread History

---

# Why RabbitMQ?

RabbitMQ provides:

- Asynchronous Processing
- Reliable Messaging
- Retry Support
- High Throughput
- Loose Coupling

---

# Design Principles

- Event-Driven Architecture
- Serverless AI Integration
- Multi-Tenant Isolation
- RAG-Based Validation
- Structured AI Output
- Human-in-the-Loop
- Tool-Based Execution
- Stateless AI Service with Persistent Conversation State

---

# Interview Questions

## Q1. Why did you use LangChain?

**Answer**

LangChain simplified prompt management, Amazon Bedrock integration, structured output parsing, RAG orchestration, and tool calling. It reduced custom orchestration code and made the workflow modular and maintainable.

---

## Q2. Why use Qdrant instead of directly asking the LLM?

**Answer**

LLMs may hallucinate or generate inconsistent entity names. Qdrant performs semantic search over enterprise master data and uses metadata filtering (Tenant ID) to retrieve only valid entities, ensuring accurate downstream API calls.

---

## Q3. Why Human-in-the-Loop?

**Answer**

Business-critical operations should not rely on guessed information. If mandatory fields are missing, the workflow pauses, requests clarification via email, and resumes from the saved conversation state after the customer replies.

---

## Q4. Why RabbitMQ?

**Answer**

RabbitMQ decouples email ingestion from AI processing, provides reliable delivery, supports retries, and enables multiple AI consumers to scale horizontally.

---

## 90-Second Interview Explanation

> "Our AutoShift platform is an event-driven, multi-tenant AI solution built using RabbitMQ, Python, LangChain, Amazon Bedrock, Qdrant, and existing .NET APIs. When a customer sends an email, it is published to RabbitMQ. A Python AI consumer processes the message, parses the email, and uses LangChain to orchestrate the workflow. LangChain constructs the system and user prompts and sends them to Amazon Bedrock (Claude). The LLM identifies the business intent—such as Create Shift, Cancel Shift, or Restart Shift—and returns structured JSON containing fields like client, location, qualification, date, time, and shift count. Before executing any business operation, we validate these entities through a RAG pipeline backed by Qdrant. Master data from existing .NET APIs is embedded and indexed in Qdrant, and retrieval uses semantic search combined with metadata filtering based on Tenant ID to ensure tenant isolation. After successful validation, LangChain invokes the appropriate tool, which calls the downstream .NET API to execute the requested shift operation. If mandatory information is missing, the workflow enters a Human-in-the-Loop stage where a clarification email is sent. When the customer replies, the message is consumed again from RabbitMQ, the conversation state is restored using the email thread stored in Qdrant, and processing resumes from the previous state instead of restarting. This architecture provides scalability, reliability, multi-tenancy, and accurate AI-driven automation while preserving all core business logic within the existing .NET platform."